In [1]:
import geopandas as gpd
import pandas as pd
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
import os
from helpers.helpers import (calculate_canopy_for_all_reaches, process_canopy_chunk)
import numpy as np
import os, time

In [2]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)

canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/property.gpkg')
access_points = gpd.read_file('output/property_accesspoints.gpkg', engine='pyogrio')

# boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('ilam')]

property = gpd.clip(property_full, sa2_chc)
TARGET_CRS = 2193

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
property_geom = property[['property_id', 'geometry']].copy()

property_geom = gpd.GeoDataFrame(
  property_geom,
  crs=property.crs,
  geometry="geometry"
)

access_points = access_points.rename(columns={'geometry': 'access_point'})

property_reaches = property_geom.merge(
    access_points[['property_id', 'access_point']],    
    on='property_id',
    how='left'
)

REACHES = [75]
# REACHES = [50, 100, 150, 200, 250, 300, 350, 400]

for distance in REACHES:
  path = f'output/property_reach_{distance}m.gpkg'
  reach_col = f'reach_{distance}m'
  
  if os.path.exists(path):
      reach_gdf = gpd.read_file(path, engine='pyogrio')
      reach_gdf = reach_gdf.rename(columns={'geometry': reach_col})
      
      property_reaches = property_reaches.merge(
          reach_gdf[['property_id', reach_col]],    
          on='property_id',
          how='left'
      )

In [4]:
n_cores = cpu_count()
n_chunks = n_cores * 2

chunks = np.array_split(property_reaches, n_chunks)

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'GeoDataFrame.transpose' instead.
  return bound(*args, **kwds)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'GeoDataFrame.swap

In [5]:
canopy_sindex = canopy.sindex
reaches = [75]
# reaches = [50, 100, 150, 200, 250, 300, 350, 400]

if __name__ == "__main__":
    with Pool(processes=n_cores) as pool:
        func = partial(
            process_canopy_chunk,
            canopy_gdf=canopy,
            canopy_sindex=canopy_sindex,
            reaches=reaches,
            buffer_dist=10
        )

        results = list(
            tqdm(
                pool.imap(func, chunks),
                total=len(chunks),
                desc="Parallel canopy calculation"
            )
        )


Parallel canopy calculation:   0%|          | 0/20 [00:00<?, ?it/s]

[PID 6530] 20/616 rows | 0.2s elapsed | 111.73 rows/s
[PID 6530] 40/616 rows | 0.3s elapsed | 155.75 rows/s
[PID 6530] 60/616 rows | 0.4s elapsed | 137.03 rows/s
[PID 6530] 80/616 rows | 0.5s elapsed | 158.72 rows/s
[PID 6530] 100/616 rows | 0.6s elapsed | 181.72 rows/s
[PID 6530] 120/616 rows | 0.6s elapsed | 206.15 rows/s
[PID 6530] 140/616 rows | 0.6s elapsed | 233.07 rows/s
[PID 6530] 160/616 rows | 0.6s elapsed | 250.46 rows/s
[PID 6530] 180/616 rows | 0.7s elapsed | 274.88 rows/s
[PID 6530] 200/616 rows | 0.7s elapsed | 292.71 rows/s
[PID 6530] 220/616 rows | 0.7s elapsed | 304.25 rows/s
[PID 6530] 240/616 rows | 0.8s elapsed | 311.83 rows/s
[PID 6530] 260/616 rows | 0.8s elapsed | 318.42 rows/s
[PID 6530] 280/616 rows | 0.9s elapsed | 328.28 rows/s
[PID 6530] 300/616 rows | 0.9s elapsed | 330.84 rows/s
[PID 6530] 320/616 rows | 0.9s elapsed | 346.33 rows/s
[PID 6530] 340/616 rows | 0.9s elapsed | 359.72 rows/s
[PID 6530] 360/616 rows | 1.0s elapsed | 369.22 rows/s
[PID 6530] 380

Parallel canopy calculation:   5%|▌         | 1/20 [00:41<13:16, 41.94s/it]

[PID 6531] 20/616 rows | 0.3s elapsed | 73.40 rows/s
[PID 6531] 40/616 rows | 0.4s elapsed | 110.71 rows/s
[PID 6531] 60/616 rows | 0.4s elapsed | 141.39 rows/s
[PID 6531] 80/616 rows | 0.4s elapsed | 180.04 rows/s
[PID 6531] 100/616 rows | 0.5s elapsed | 207.09 rows/s
[PID 6531] 120/616 rows | 0.5s elapsed | 238.62 rows/s
[PID 6531] 140/616 rows | 0.5s elapsed | 266.77 rows/s
[PID 6531] 160/616 rows | 0.5s elapsed | 301.71 rows/s
[PID 6531] 180/616 rows | 0.5s elapsed | 331.06 rows/s
[PID 6531] 200/616 rows | 0.6s elapsed | 355.80 rows/s
[PID 6531] 220/616 rows | 0.6s elapsed | 359.78 rows/s
[PID 6531] 240/616 rows | 0.7s elapsed | 357.98 rows/s
[PID 6531] 260/616 rows | 0.7s elapsed | 366.31 rows/s
[PID 6531] 280/616 rows | 0.8s elapsed | 364.20 rows/s
[PID 6531] 300/616 rows | 0.9s elapsed | 346.63 rows/s
[PID 6531] 320/616 rows | 0.9s elapsed | 359.33 rows/s
[PID 6531] 340/616 rows | 0.9s elapsed | 372.36 rows/s
[PID 6531] 360/616 rows | 0.9s elapsed | 382.74 rows/s
[PID 6531] 380/

Parallel canopy calculation:  10%|█         | 2/20 [01:34<14:32, 48.44s/it]

...

Parallel canopy calculation:  90%|█████████ | 18/20 [23:33<02:55, 87.81s/it]

[PID 6537] 20/615 rows | 0.5s elapsed | 40.69 rows/s
[PID 6537] 40/615 rows | 0.5s elapsed | 74.68 rows/s
[PID 6537] 60/615 rows | 0.6s elapsed | 106.22 rows/s
[PID 6537] 80/615 rows | 0.6s elapsed | 134.84 rows/s
[PID 6537] 100/615 rows | 0.6s elapsed | 161.07 rows/s
[PID 6537] 120/615 rows | 0.6s elapsed | 186.25 rows/s
[PID 6537] 140/615 rows | 0.7s elapsed | 210.62 rows/s
[PID 6537] 160/615 rows | 0.7s elapsed | 230.07 rows/s
[PID 6537] 180/615 rows | 0.7s elapsed | 249.96 rows/s
[PID 6537] 200/615 rows | 0.8s elapsed | 266.34 rows/s
[PID 6537] 220/615 rows | 0.8s elapsed | 285.75 rows/s
[PID 6537] 240/615 rows | 0.8s elapsed | 301.15 rows/s
[PID 6537] 260/615 rows | 0.8s elapsed | 319.82 rows/s
[PID 6537] 280/615 rows | 0.9s elapsed | 325.75 rows/s
[PID 6537] 300/615 rows | 0.9s elapsed | 343.78 rows/s
[PID 6537] 320/615 rows | 0.9s elapsed | 356.08 rows/s
[PID 6537] 340/615 rows | 0.9s elapsed | 375.20 rows/s
[PID 6537] 360/615 rows | 0.9s elapsed | 387.62 rows/s
[PID 6537] 380/6

Parallel canopy calculation:  95%|█████████▌| 19/20 [24:58<01:27, 87.13s/it]

[PID 6539] 20/615 rows | 0.6s elapsed | 34.43 rows/s
[PID 6539] 40/615 rows | 0.7s elapsed | 61.53 rows/s
[PID 6539] 60/615 rows | 0.7s elapsed | 86.96 rows/s
[PID 6539] 80/615 rows | 0.7s elapsed | 110.57 rows/s
[PID 6539] 100/615 rows | 0.8s elapsed | 132.28 rows/s
[PID 6539] 120/615 rows | 0.8s elapsed | 153.29 rows/s
[PID 6539] 140/615 rows | 0.8s elapsed | 172.41 rows/s
[PID 6539] 160/615 rows | 0.8s elapsed | 188.75 rows/s
[PID 6539] 180/615 rows | 0.9s elapsed | 204.44 rows/s
[PID 6539] 200/615 rows | 0.9s elapsed | 217.99 rows/s
[PID 6539] 220/615 rows | 1.0s elapsed | 230.13 rows/s
[PID 6539] 240/615 rows | 1.0s elapsed | 240.27 rows/s
[PID 6539] 260/615 rows | 1.0s elapsed | 252.59 rows/s
[PID 6539] 280/615 rows | 1.1s elapsed | 261.67 rows/s
[PID 6539] 300/615 rows | 1.1s elapsed | 270.05 rows/s
[PID 6539] 320/615 rows | 1.1s elapsed | 281.79 rows/s
[PID 6539] 340/615 rows | 1.2s elapsed | 290.10 rows/s
[PID 6539] 360/615 rows | 1.2s elapsed | 299.19 rows/s
[PID 6539] 380/61

Parallel canopy calculation: 100%|██████████| 20/20 [25:42<00:00, 77.12s/it]


In [6]:
for chunk_result in results:
    for idx, res in chunk_result:
        if res is None:
            continue
        for key, value in res.items():
            property_reaches.at[idx, key] = value


In [7]:
# reaches = [50, 100, 150, 200, 250, 300, 350, 400]
reaches = [75]

canopy_cols = [f"canopy_{d}m" for d in reaches]

canopy_out = property_reaches[canopy_cols].copy()

canopy_out["property_id"] = property_reaches.index

canopy_out = canopy_out[["property_id"] + canopy_cols]

canopy_out = gpd.GeoDataFrame(
    canopy_out,
    geometry=property_reaches.geometry,
    crs=property_reaches.crs
)


In [8]:
canopy_out.head()

,property_id,canopy_75m,geometry
0,0,71.199176,POINT (1564919.129 5174157.305)
1,1,39.233007,POINT (1565047.92 5174178.753)
2,2,153.781247,POINT (1564904.931 5174179.536)
3,3,32.559216,POINT (1565029.188 5174201.753)
4,4,162.647867,POINT (1564988.184 5174211.698)


In [9]:
out_path = "output/property_isodistance_canopies_75.gpkg"

canopy_out.to_file(
    out_path,
    layer="canopy_by_reach",
    driver="GPKG",
    engine="pyogrio"
)

In [10]:
canopy_out.shape

(12317, 3)

In [11]:
canopy_out.head()

,property_id,canopy_75m,geometry
0,0,71.199176,POINT (1564919.129 5174157.305)
1,1,39.233007,POINT (1565047.92 5174178.753)
2,2,153.781247,POINT (1564904.931 5174179.536)
3,3,32.559216,POINT (1565029.188 5174201.753)
4,4,162.647867,POINT (1564988.184 5174211.698)


# combine all canopy files

In [12]:
property_wo_75 = gpd.read_file('output/property_isodistance_canopies.gpkg')

In [13]:
print(canopy_out.shape)
print(property_wo_75.shape)


(12317, 3)
(12317, 11)


In [14]:
property_canopies_full = property_wo_75.merge(
  canopy_out[['property_id', 'canopy_75m']],
  how='left',
  on='property_id'
)

In [15]:
property_canopies_full.shape

(12317, 12)

In [16]:
property_canopies_full.head()

,property_id,canopy_50m,canopy_100m,canopy_150m,canopy_200m,canopy_250m,canopy_300m,canopy_350m,canopy_400m,canopy_25m,geometry,canopy_75m
0,0,25.782986,192.711307,204.768643,319.550849,550.032059,682.414635,932.266395,1242.955144,14.466409,POINT (1564919.129 5174157.305),71.199176
1,1,30.495473,116.039225,369.255412,501.136692,852.088438,1146.041794,1603.443922,2132.706457,27.566893,POINT (1565047.92 5174178.753),39.233007
2,2,146.128014,161.708964,190.194919,204.768643,251.751149,496.983665,679.576348,856.535574,32.582817,POINT (1564904.931 5174179.536),153.781247
3,3,30.495473,165.048229,353.665378,511.542446,909.228096,1135.821806,1627.178436,2312.325189,0.000000,POINT (1565029.188 5174201.753),32.559216
4,4,42.975251,188.162874,194.655967,222.928693,385.847184,632.951004,767.283211,1057.061872,9.263134,POINT (1564988.184 5174211.698),162.647867


In [17]:
property_canopies_full.to_file(
    'output/property_isodistance_canopies.gpkg',
    layer="canopy_by_reach",
    driver="GPKG",
    engine="pyogrio"
)